# 02_ex_<agreement>_<topic> — exploration and approval drafting

Use this notebook to profile source data, draft AI-assisted suggestions, and document human-reviewed decisions before anything is enforced in pipeline execution.

**You edit**
- Agreement/topic metadata
- Source target details
- Draft contract, DQ, and classification decisions

**This notebook produces**
- Profiling evidence
- Candidate DQ/classification suggestions
- Human-reviewed contract draft and approved decisions


## Segment 1: Load shared config and imports

What this does: loads `00_env_config` and imports exploration helpers.

Functions used: `load_fabric_config`, `build_runtime_context`, `generate_metadata_profile`, `generate_dq_rule_candidates_with_fabric_ai`.

Output: configured runtime context and helper availability.


In [ ]:
%run 00_env_config


In [ ]:
from fabricops_kit.environment_config import bootstrap_fabric_env, get_path, load_fabric_config
from fabricops_kit.fabric_input_output import lakehouse_table_read, warehouse_read
from fabricops_kit.runtime_context import assert_notebook_name_valid, build_runtime_context, validate_notebook_name
from fabricops_kit.data_profiling import build_ai_quality_context, generate_metadata_profile, profile_dataframe_to_metadata
from fabricops_kit.data_quality import (
    build_manual_dq_rule_prompt_package,
    generate_dq_rule_candidates_with_fabric_ai,
    suggest_dq_rules_prompt,
)
from fabricops_kit.data_governance import (
    build_governance_classification_records,
    build_manual_governance_prompt_package,
    classify_columns,
    generate_governance_candidates_with_fabric_ai,
    summarize_governance_classifications,
)
from fabricops_kit.data_contracts import validate_contract_dict, write_contract_to_lakehouse


## Agreement and approved usage


In [ ]:
DATA_AGREEMENT = "TODO_replace_agreement"
APPROVED_USAGE = "TODO_replace_approved_usage"
ENV_NAME = "dev"
SOURCE_LAYER = "source"
SOURCE_KIND = "lakehouse"
SOURCE_TABLE = "TODO_source_table"


In [ ]:
CONFIG = load_fabric_config(CONFIG)
validate_notebook_name()
assert_notebook_name_valid()
runtime_context = build_runtime_context(dataset_name="topic_dataset", environment=ENV_NAME, source_table=SOURCE_TABLE, target_table="unified.topic_dataset")
bootstrap_fabric_env(config=CONFIG, env=ENV_NAME, required_targets=["source", "unified", "product"])


In [ ]:
source_path = get_path(ENV_NAME, SOURCE_LAYER, config=CONFIG)


In [ ]:
if SOURCE_KIND == "lakehouse":
    df_source = lakehouse_table_read(source_path, SOURCE_TABLE)
else:
    df_source = warehouse_read(env=ENV_NAME, target=SOURCE_LAYER, schema="dbo", table=SOURCE_TABLE, config=CONFIG)


In [ ]:
print(df_source.schema)
display(df_source.limit(20))


## Segment 2: Profile source and capture evidence

What this does: loads source data and builds profiling metadata for review.

Functions used: `lakehouse_table_read`/`warehouse_read`, `generate_metadata_profile`, `profile_dataframe_to_metadata`.

Output: source profile rows for human review and downstream drafting.


In [ ]:
profile_rows = generate_metadata_profile(df_source, dataset_name="topic_dataset")
metadata_rows = profile_dataframe_to_metadata(df_source, dataset_name="topic_dataset")
display(profile_rows)
display(metadata_rows)



# AI-assisted DQ suggestions (candidate-only; requires human approval before enforcement)
prompt = suggest_dq_rules_prompt(
    profile_df=profile_rows,
    table_name="EMAIL_LOGS",
    business_context="Describe what this table is used for.",
    output_format="python_config",
    prompt_template=DQ_RULE_SUGGESTION_PROMPT_TEMPLATE,
)
print(prompt)


## Exploration findings
Document EDA findings here.


## Transformation rationale
Document approved transformation rationale for pipeline handoff.


## Segment 3: AI-assisted drafting (advisory only)

What this does: creates prompt context and candidate DQ/governance suggestions.

Functions used: `build_ai_quality_context`, `build_manual_dq_rule_prompt_package`, `generate_dq_rule_candidates_with_fabric_ai`, `generate_governance_candidates_with_fabric_ai`.

Output: candidate suggestions that require human approval.


In [ ]:
ai_context = build_ai_quality_context(profile_rows, dataset_name="topic_dataset")
print("AI advisory only; human approval required.")
# Optional:




## Draft source input contract

This notebook helps humans define the **source input contract** for the pipeline.

- Capture required columns, optional columns, business keys, classifications, and approved DQ rules.
- In Fabric-first usage, draft this as a Python dict or small table, then write approved values to metadata tables.
- AI suggestions are advisory only and require human/steward approval.
- YAML/ODCS export is optional and will be supported in a follow-up PR.


In [ ]:
SOURCE_INPUT_CONTRACT_DRAFT = {
    "contract_id": "TODO_contract_id",
    "contract_type": "source_input",
    "dataset_name": "topic_dataset",
    "object_name": SOURCE_TABLE,
    "version": "0.1.0",
    "status": "draft",
    "required_columns": [],
    "optional_columns": [],
    "business_keys": [],
    "classifications": {},
    "quality_rules": [],
    "approved_by": None,
    "approval_note": None,
}
print(SOURCE_INPUT_CONTRACT_DRAFT)


## Human-reviewed DQ decisions
Copy only approved deterministic rules to 03_pc notebook.


In [ ]:
classification_candidates = classify_columns(
    profile_rows,
    business_context={"agreement": DATA_AGREEMENT},
)
print(summarize_governance_classifications(classification_candidates))
# Optional:



## Segment 4: Human approval and contract write

What this does: captures approved classifications/rules and writes approved contract metadata.

Functions used: `classify_columns`, `summarize_governance_classifications`, `validate_contract_dict`, `write_contract_to_lakehouse`.

Output: approved contract and governance evidence for pipeline enforcement.


## Human-reviewed classification decisions
Approve labels with governance stewards.


In [ ]:
APPROVED_CLASSIFICATIONS = []
classification_records = build_governance_classification_records(dataset_name="topic_dataset", environment=ENV_NAME, classifications=APPROVED_CLASSIFICATIONS, run_id=runtime_context["run_id"])
display(classification_records)


## Approve and write source input contract
Validate draft contracts, then explicitly gate writes to the dedicated metadata target.


In [ ]:
metadata_path = get_path(ENV_NAME, "metadata", config=CONFIG)

contract_errors = validate_contract_dict(SOURCE_INPUT_CONTRACT_DRAFT)
if contract_errors:
    raise ValueError("Contract validation failed: " + " | ".join(contract_errors))

SOURCE_INPUT_CONTRACT_APPROVED = {
    **SOURCE_INPUT_CONTRACT_DRAFT,
    "status": "approved",
    "approved_by": "TODO_APPROVER",
    "approval_note": "TODO_APPROVAL_NOTE",
}

WRITE_APPROVED_CONTRACT = False
if WRITE_APPROVED_CONTRACT:
    write_contract_to_lakehouse(SOURCE_INPUT_CONTRACT_APPROVED, metadata_path)


In [ ]:
from fabricops_kit.data_lineage import build_lineage_from_notebook_code

# Optional documentation helper only (not transformation logic).
# Paste or load exported notebook source when ready to document lineage.
# result = build_lineage_from_notebook_code(notebook_code)
# display(result["deterministic_steps"])
# print(result["copilot_prompt"])
